# Personalized Door State Classifier — Experiment Notebook

This notebook records the training and evaluation workflow for a personalized EfficientNet-B0 model that classifies doors as `Closed` or `Open` (`Open` includes partially open). It accompanies a lightweight GitHub project report; it is not a general-purpose inference package.

**Data and checkpoint are private and are not included.** To rerun the experiment, configure the paths in the setup cell and provide the expected `train/val/test` folders and initial checkpoint. The reported validation result is 97.01%; results should not be interpreted as a guarantee on unseen doors.


In [ ]:
# Google Colab only. If you run this notebook outside Colab, skip this cell
# and update the paths in the configuration cell below.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b0
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score

# Reproducibility (some GPU operations may remain nondeterministic).
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ---- Update these paths for your environment ----
DRIVE_ROOT = Path('/content/drive/MyDrive')
DATASET_ROOT = DRIVE_ROOT / 'door_final_dataset'
INITIAL_CHECKPOINT_PATH = DRIVE_ROOT / 'best_door_classifier.pth'
BEST_CHECKPOINT_PATH = DRIVE_ROOT / 'best_door_classifier_finetuned.pth'

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16
NUM_EPOCHS = 15
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PIN_MEMORY = DEVICE.type == 'cuda'
CLASS_NAMES = ['Closed', 'Open']  # ImageFolder sorts class folder names alphabetically.

print(f'Device: {DEVICE}')
print(f'Dataset root: {DATASET_ROOT}')
print(f'Initial checkpoint: {INITIAL_CHECKPOINT_PATH}')
print(f'Best checkpoint output: {BEST_CHECKPOINT_PATH}')

In [ ]:
# Check that the required files and folders exist before training starts.
required_dirs = [DATASET_ROOT / split for split in ('train', 'val', 'test')]
missing_dirs = [str(path) for path in required_dirs if not path.is_dir()]
if missing_dirs:
    raise FileNotFoundError(
        'Missing dataset split folder(s):\n' + '\n'.join(missing_dirs) +
        '\nExpected: door_final_dataset/{train,val,test}/{Closed,Open}/...'
    )
if not INITIAL_CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        f'Initial checkpoint not found: {INITIAL_CHECKPOINT_PATH}\n'
        'Update INITIAL_CHECKPOINT_PATH to the trained checkpoint used by your current workflow.'
    )

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
])

eval_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(DATASET_ROOT / 'train', transform=train_transform)
val_dataset = datasets.ImageFolder(DATASET_ROOT / 'val', transform=eval_transform)
test_dataset = datasets.ImageFolder(DATASET_ROOT / 'test', transform=eval_transform)

for split_name, dataset in [('train', train_dataset), ('val', val_dataset), ('test', test_dataset)]:
    if dataset.classes != CLASS_NAMES:
        raise ValueError(
            f'{split_name} class folders are {dataset.classes}, expected {CLASS_NAMES}. '
            'Ensure every split has the same Closed and Open directory names.'
        )

print('Class-to-index mapping:', train_dataset.class_to_idx)
for split_name, dataset in [('Train', train_dataset), ('Validation', val_dataset), ('Test', test_dataset)]:
    counts = {name: 0 for name in dataset.classes}
    for _, class_idx in dataset.samples:
        counts[dataset.classes[class_idx]] += 1
    print(f'{split_name}: {len(dataset):,} images | {counts}')

In [ ]:
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)

batch_images, batch_labels = next(iter(train_loader))
print('Image batch:', tuple(batch_images.shape))
print('Label batch:', tuple(batch_labels.shape))

In [ ]:
# Build the same architecture as the original model and load its initial weights.
model = efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(CLASS_NAMES))

try:
    checkpoint = torch.load(INITIAL_CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
except TypeError:  # Compatibility with older PyTorch releases.
    checkpoint = torch.load(INITIAL_CHECKPOINT_PATH, map_location=DEVICE)

# Accept either a plain state_dict or a wrapped checkpoint.
if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    checkpoint = checkpoint['model_state_dict']
model.load_state_dict(checkpoint)
model = model.to(DEVICE)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f'Trainable parameters: {trainable_params:,}')
print(f'Total parameters: {total_params:,}')

In [ ]:
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss()


def run_epoch(loader, training):
    """Run one training or evaluation epoch; metrics are weighted by image count."""
    model.train(training)
    total_loss = 0.0
    total_correct = 0
    total_images = 0

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_images += batch_size

    if total_images == 0:
        raise ValueError('The data loader is empty.')
    return total_loss / total_images, total_correct / total_images

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_accuracy = -1.0
best_epoch = None
BEST_CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_accuracy = run_epoch(train_loader, training=True)
    val_loss, val_accuracy = run_epoch(val_loader, training=False)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_accuracy)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_accuracy)

    print(
        f'Epoch [{epoch:02d}/{NUM_EPOCHS}] '
        f'train_loss={train_loss:.4f} train_acc={train_accuracy:.2%} '
        f'val_loss={val_loss:.4f} val_acc={val_accuracy:.2%}'
    )

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_epoch = epoch
        torch.save(model.state_dict(), BEST_CHECKPOINT_PATH)
        print(f'  Saved best checkpoint: {BEST_CHECKPOINT_PATH}')

print(f'Best validation accuracy: {best_val_accuracy:.2%} (epoch {best_epoch})')

In [ ]:
# Always evaluate the best checkpoint, not merely the weights from the last epoch.
try:
    best_state = torch.load(BEST_CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
except TypeError:
    best_state = torch.load(BEST_CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(best_state)
model.eval()

test_predictions = []
test_labels = []
with torch.inference_mode():
    for images, labels in test_loader:
        images = images.to(DEVICE, non_blocking=True)
        logits = model(images)
        predictions = logits.argmax(dim=1).cpu().numpy()
        test_predictions.extend(predictions.tolist())
        test_labels.extend(labels.numpy().tolist())

if not test_labels:
    raise ValueError('The test split is empty.')

test_accuracy = accuracy_score(test_labels, test_predictions)
print(f'Test accuracy: {test_accuracy:.2%}')
print('\nClassification report:')
print(classification_report(
    test_labels, test_predictions,
    labels=[0, 1], target_names=CLASS_NAMES, zero_division=0, digits=4
))

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, len(history['train_loss']) + 1)
axes[0].plot(epochs, history['train_loss'], label='Train')
axes[0].plot(epochs, history['val_loss'], label='Validation')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs, history['train_acc'], label='Train')
axes[1].plot(epochs, history['val_acc'], label='Validation')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylim(0, 1)
axes[1].legend()
plt.tight_layout()
plt.show()

cm = confusion_matrix(test_labels, test_predictions, labels=[0, 1])
display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
display.plot(values_format='d', cmap='Blues')
plt.title('Test Confusion Matrix')
plt.tight_layout()
plt.show()

## Evaluation note

Report validation and test results separately. For a reliable estimate on new doors, keep entire recording sessions or days out of training and validation; random splits of near-identical video frames can inflate the measured accuracy. The custom images and checkpoints are not included in this repository.